## Market Baseket Analysis 

Market Basket Analysis is a data-driven technique used to uncover patterns and relationships within large transactional datasets. 

Market Basket Analysis is a valuable tool for businesses seeking to optimize their product offerings, increase cross-selling opportunities, and improve marketing strategies. It can lead to higher revenue, enhanced customer satisfaction, and overall business success.

# 1. Import Libraries

In [5]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

# 2. Load and Inspect Data

In [8]:
data = pd.read_csv("D://Anaconda//Market Baseket//market_basket_dataset.csv")
display(data.head())
display(data.describe())
display(data.isnull().sum())

,BillNo,Itemname,Quantity,Price,CustomerID
0,1000,Apples,5,8.30,52299
1,1000,Butter,4,6.06,11752
2,1000,Eggs,4,2.66,16415
3,1000,Potatoes,4,8.10,22889
4,1004,Oranges,2,7.26,52255


,BillNo,Quantity,Price,CustomerID
count,500.000000,500.000000,500.000000,500.000000
mean,1247.442000,2.978000,5.617660,54229.800000
std,144.483097,1.426038,2.572919,25672.122585
min,1000.000000,1.000000,1.040000,10504.000000
25%,1120.000000,2.000000,3.570000,32823.500000
50%,1246.500000,3.000000,5.430000,53506.500000
75%,1370.000000,4.000000,7.920000,76644.250000
max,1497.000000,5.000000,9.940000,99162.000000


BillNo        0
Itemname      0
Quantity      0
Price         0
CustomerID    0
dtype: int64

# 3. Item Distribution

In [9]:
fig = px.histogram(data, x='Itemname', title='Item Distribution')
fig.show()

# 4. Top 10 Popular Items

In [10]:
item_popularity = data.groupby('Itemname')['Quantity'].sum().sort_values(ascending=False)
top_n = 10

fig = go.Figure()
fig.add_trace(go.Bar(x=item_popularity.index[:top_n], y=item_popularity.values[:top_n],
                     text=item_popularity.values[:top_n], textposition='auto',
                     marker=dict(color='skyblue')))
fig.update_layout(title=f'Top {top_n} Most Popular Items',
                  xaxis_title='Item Name', yaxis_title='Total Quantity Sold')
fig.show()

# 5. Customer Behaviour

In [11]:
customer_behavior = data.groupby('CustomerID').agg({'Quantity': 'mean', 'Price': 'sum'}).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=customer_behavior['Quantity'], y=customer_behavior['Price'],
    mode='markers', text=customer_behavior['CustomerID'],
    marker=dict(size=10, color='coral')))
fig.update_layout(title='Customer Behavior',
                  xaxis_title='Average Quantity', yaxis_title='Total Spending')
fig.show()

# 6. Prepare Transactions

In [12]:
transactions = data.groupby('BillNo')['Itemname'].apply(list).tolist()
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# 7. Aprior Algorithm

In [13]:
frequent_itemsets_ap = apriori(df_encoded, min_support=0.01, use_colnames=True)
rules_ap = association_rules(frequent_itemsets_ap, metric="lift", min_threshold=0.5)

# Convert frozensets to strings for display
rules_ap['antecedents'] = rules_ap['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_ap['consequents'] = rules_ap['consequents'].apply(lambda x: ', '.join(list(x)))

display(rules_ap[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

,antecedents,consequents,support,confidence,lift
0,Bread,Apples,0.045752,0.304348,1.862609
1,Apples,Bread,0.045752,0.280000,1.862609
2,Apples,Butter,0.026144,0.160000,0.979200
3,Butter,Apples,0.026144,0.160000,0.979200
4,Apples,Cereal,0.019608,0.120000,0.592258
5,Cereal,Apples,0.019608,0.096774,0.592258
6,Apples,Cheese,0.039216,0.240000,1.311429
7,Cheese,Apples,0.039216,0.214286,1.311429
8,Apples,Chicken,0.032680,0.200000,1.530000
9,Chicken,Apples,0.032680,0.250000,1.530000


# 8. FP-Growth Algorithm

In [14]:
frequent_itemsets_fp = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)
rules_fp = association_rules(frequent_itemsets_fp, metric="lift", min_threshold=0.5)

rules_fp['antecedents'] = rules_fp['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_fp['consequents'] = rules_fp['consequents'].apply(lambda x: ', '.join(list(x)))

display(rules_fp[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


,antecedents,consequents,support,confidence,lift
0,Potatoes,Tomatoes,0.058824,0.333333,1.888889
1,Tomatoes,Potatoes,0.058824,0.333333,1.888889
2,Potatoes,Cereal,0.045752,0.259259,1.279570
3,Cereal,Potatoes,0.045752,0.225806,1.279570
4,Oranges,Potatoes,0.065359,0.344828,1.954023
5,Potatoes,Oranges,0.065359,0.370370,1.954023
6,Potatoes,Bananas,0.065359,0.370370,1.531532
7,Bananas,Potatoes,0.065359,0.270270,1.531532
8,Coffee,Potatoes,0.052288,0.242424,1.373737
9,Potatoes,Coffee,0.052288,0.296296,1.373737


# 9. Visualize Rules

In [15]:
def visualize_rules(rules, title):
    fig = px.scatter(rules, x='support', y='confidence', size='lift',
                     hover_data=['antecedents', 'consequents'],
                     title=title, color='lift', color_continuous_scale='Viridis')
    fig.show()

visualize_rules(rules_ap, "Association Rules - Apriori")
visualize_rules(rules_fp, "Association Rules - FP-Growth")
